# sql-generator: Natural Language to SQL with Claude API

This notebook demonstrates the NL-to-SQL pipeline:
1. Load the Chinook SQLite schema
2. Ask 5 natural language questions
3. Generate SQL via the Claude API (Haiku)
4. Execute and evaluate the results

**Requirements:**
- `ANTHROPIC_API_KEY` must be set in the environment
- `sql-generator/data/chinook/Chinook.sqlite` must exist

**⚠️ HUMAN APPROVAL REQUIRED** before running Cell 5 (API calls). Each question triggers one Claude API call.

In [1]:
import pathlib
import sys
import os
from IPython.display import display
import pandas as pd

# Locate sql-generator root regardless of where the notebook is run from
_here = pathlib.Path().resolve()
if (_here / 'sql-generator').exists():      # repo root
    SQL_ROOT = _here / 'sql-generator'
elif (_here / 'src').exists():              # sql-generator/
    SQL_ROOT = _here
elif (_here.parent / 'src').exists():       # sql-generator/notebooks/
    SQL_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate sql-generator root from {_here}')

SRC = SQL_ROOT / 'src'
DB_PATH = SQL_ROOT / 'data' / 'chinook' / 'Chinook.sqlite'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f'SQL_ROOT : {SQL_ROOT}')
print(f'DB exists: {DB_PATH.exists()} ({DB_PATH})')
print(f'API key  : {"set" if os.environ.get("ANTHROPIC_API_KEY") else "NOT SET — set before running Cell 5"}')

SQL_ROOT : /Users/rajeevkulkarni/ml-explorations/sql-generator
DB exists: True (/Users/rajeevkulkarni/ml-explorations/sql-generator/data/chinook/Chinook.sqlite)
API key  : set


In [2]:
from schema_loader import load_schema, format_schema_context

schema = load_schema(DB_PATH)
schema_context = format_schema_context(schema)

print(f'Tables ({len(schema)}): {sorted(schema.keys())}')
print()
print('--- Schema context (first 1500 chars) ---')
print(schema_context[:1500])

Tables (11): ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']

--- Schema context (first 1500 chars) ---
TABLE Album
  Columns: AlbumId (INTEGER), Title (NVARCHAR(160)), ArtistId (INTEGER)
  Sample rows:
    [1, 'For Those About To Rock We Salute You', 1]
    [2, 'Balls to the Wall', 2]
    [3, 'Restless and Wild', 2]

TABLE Artist
  Columns: ArtistId (INTEGER), Name (NVARCHAR(120))
  Sample rows:
    [1, 'AC/DC']
    [2, 'Accept']
    [3, 'Aerosmith']

TABLE Customer
  Columns: CustomerId (INTEGER), FirstName (NVARCHAR(40)), LastName (NVARCHAR(20)), Company (NVARCHAR(80)), Address (NVARCHAR(70)), City (NVARCHAR(40)), State (NVARCHAR(40)), Country (NVARCHAR(40)), PostalCode (NVARCHAR(10)), Phone (NVARCHAR(24)), Fax (NVARCHAR(24)), Email (NVARCHAR(60)), SupportRepId (INTEGER)
  Sample rows:
    [1, 'Luís', 'Gonçalves', 'Embraer - Empresa Brasileira de Aeronáutica S.A.', 'Av. Brigadeiro Faria Lima, 2170', 'S

## 5 Natural Language Questions

1. List all artists alphabetically
2. How many tracks are in each genre? Order by count descending.
3. Who are the top 5 customers by total purchase amount?
4. Which albums have more than 10 tracks? Show album title and track count.
5. What is the average track length in minutes per media type?

In [3]:
# HUMAN APPROVAL REQUIRED — this cell calls the Claude API (5 requests)
from generator import generate_sql
from executor import execute_query

questions = [
    "List all artists alphabetically",
    "How many tracks are in each genre? Order by count descending.",
    "Who are the top 5 customers by total purchase amount?",
    "Which albums have more than 10 tracks? Show album title and track count.",
    "What is the average track length in minutes per media type?",
]

results = []
for i, question in enumerate(questions, 1):
    print(f'\n--- Q{i}: {question} ---')
    try:
        sql = generate_sql(question, schema_context)
        print(f'SQL: {sql}')
        df = execute_query(sql, DB_PATH)
        display(df.head(10))
        results.append({'question': question, 'sql': sql, 'rows': len(df), 'error': None})
    except Exception as exc:
        print(f'ERROR: {exc}')
        results.append({'question': question, 'sql': None, 'rows': None, 'error': str(exc)})


--- Q1: List all artists alphabetically ---


SQL: SELECT Name FROM Artist ORDER BY Name ASC;


,Name
0,A Cor Do Som
1,AC/DC
2,Aaron Copland & London Symphony Orchestra
3,Aaron Goldberg
4,Academy of St. Martin in the Fields & Sir Nevi...
5,Academy of St. Martin in the Fields Chamber En...
6,"Academy of St. Martin in the Fields, John Birc..."
7,"Academy of St. Martin in the Fields, Sir Nevil..."
8,"Academy of St. Martin in the Fields, Sir Nevil..."
9,Accept



--- Q2: How many tracks are in each genre? Order by count descending. ---


SQL: SELECT g.GenreId, g.Name, COUNT(t.TrackId) AS TrackCount
FROM Genre g
LEFT JOIN Track t ON g.GenreId = t.GenreId
GROUP BY g.GenreId, g.Name
ORDER BY TrackCount DESC;


...

SQL: SELECT Album.Title, COUNT(Track.TrackId) AS TrackCount
FROM Album
JOIN Track ON Album.AlbumId = Track.AlbumId
GROUP BY Album.AlbumId, Album.Title
HAVING COUNT(Track.TrackId) > 10
ORDER BY TrackCount DESC;


,Title,TrackCount
0,Greatest Hits,57
1,Minha Historia,34
2,Unplugged,30
3,"Lost, Season 3",26
4,"Lost, Season 1",25
5,"The Office, Season 3",25
6,My Way: The Best Of Frank Sinatra [Disc 1],24
7,"Lost, Season 2",24
8,"Battlestar Galactica (Classic), Season 1",24
9,Afrociberdelia,23



--- Q5: What is the average track length in minutes per media type? ---


SQL: SELECT 
  m.MediaTypeId,
  m.Name,
  ROUND(AVG(t.Milliseconds) / 60000.0, 2) AS AvgTrackLengthMinutes
FROM Track t
JOIN MediaType m ON t.MediaTypeId = m.MediaTypeId
GROUP BY m.MediaTypeId, m.Name
ORDER BY m.MediaTypeId;


,MediaTypeId,Name,AvgTrackLengthMinutes
0,1,MPEG audio file,4.43
1,2,Protected AAC audio file,4.70
2,3,Protected MPEG-4 video file,39.05
3,4,Purchased AAC audio file,4.35
4,5,AAC audio file,4.61


In [4]:
# Evaluate Q1: compare generated SQL against a hand-written reference
from evaluator import evaluate

reference_q1 = "SELECT Name FROM Artist ORDER BY Name"
generated_q1 = results[0]['sql'] if results else None

if generated_q1:
    eval_result = evaluate(generated_q1, reference_q1, DB_PATH)
    print('Q1 Evaluation:')
    print(f'  Exact match     : {eval_result["match"]}')
    print(f'  Results match   : {eval_result["results_match"]}')
    print(f'  Generated shape : {eval_result["generated_result_shape"]}')
    print(f'  Reference shape : {eval_result["reference_result_shape"]}')
    print(f'  Generated SQL   : {generated_q1}')
    print(f'  Reference SQL   : {reference_q1}')
else:
    print('Run Cell 5 first to generate SQL.')

Q1 Evaluation:
  Exact match     : False
  Results match   : True
  Generated shape : (275, 1)
  Reference shape : (275, 1)
  Generated SQL   : SELECT Name FROM Artist ORDER BY Name ASC;
  Reference SQL   : SELECT Name FROM Artist ORDER BY Name


In [5]:
# Summary of all 5 results
if results:
    summary = pd.DataFrame([
        {
            'Q': f'Q{i+1}',
            'Question': r['question'][:60] + ('...' if len(r['question']) > 60 else ''),
            'SQL generated': 'Y' if r['sql'] else 'N',
            'Row count': r['rows'],
            'Error': r['error'] if r['error'] else '',
        }
        for i, r in enumerate(results)
    ])
    display(summary)
else:
    print('Run Cell 5 first to generate results.')

,Q,Question,SQL generated,Row count,Error
0,Q1,List all artists alphabetically,Y,275,
1,Q2,How many tracks are in each genre? Order by co...,Y,25,
2,Q3,Who are the top 5 customers by total purchase ...,Y,5,
3,Q4,Which albums have more than 10 tracks? Show al...,Y,183,
4,Q5,What is the average track length in minutes pe...,Y,5,
